In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from src import utils

### data

In [ ]:
# report summary of the given files
def get_data_from_files(nseq_list, nseq_list_sub, *enrichment_files):
    df = pd.DataFrame([])
    for f in enrichment_files:
        df_i = pd.read_csv(f, sep="\t", index_col=0)
        df = pd.concat([df, df_i])
    
    df.columns = df.columns.astype(int)
    df = df.reindex(sorted(df.columns), axis=1)

    return([np.array(nseq_list), df.mean().values, np.array(nseq_list_sub), df.mean()[nseq_list_sub].values, df.sem()[nseq_list_sub].values])

In [ ]:
def get_data_sample(data_type):
    full = pd.DataFrame([])
    sub = pd.DataFrame([])

    data = get_data_from_files(nseq_list, nseq_list_sub, "/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/%s/enrichment/RankColdness_100.txt" % data_type)
    full_df = pd.DataFrame([data[0], data[1]]).transpose()
    full_df["sample"] = data_type.replace("human", "")
    full_df["rank"] = "C-score"
    sub_df = pd.DataFrame([data[2], data[3], data[4]]).transpose()
    sub_df["sample"] = data_type.replace("human", "")
    sub_df["rank"] = "C-score"
    full = pd.concat([full, full_df])
    sub = pd.concat([sub, sub_df])

    data = get_data_from_files(nseq_list, nseq_list_sub, "/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/%s/enrichment/RankSPP_100.txt" % data_type)
    full_df = pd.DataFrame([data[0], data[1]]).transpose()
    full_df["sample"] = data_type.replace("human", "")
    full_df["rank"] = "SPP"
    sub_df = pd.DataFrame([data[2], data[3], data[4]]).transpose()
    sub_df["sample"] = data_type.replace("human", "")
    sub_df["rank"] = "SPP"
    full = pd.concat([full, full_df])
    sub = pd.concat([sub, sub_df])

    data = get_data_from_files(nseq_list, nseq_list_sub, "/home/jg2447/slayman/motif_inference/result/EnrichmentGenomeShuffle/%s/enrichment/RankColdness_100.txt" % data_type)
    full_df = pd.DataFrame([data[0], data[1]]).transpose()
    full_df["sample"] = data_type.replace("human", "")
    full_df["rank"] = "C-score Shuffled"
    sub_df = pd.DataFrame([data[2], data[3], data[4]]).transpose()
    sub_df["sample"] = data_type.replace("human", "")
    sub_df["rank"] = "C-score Shuffled"
    full = pd.concat([full, full_df])
    sub = pd.concat([sub, sub_df])

    data = get_data_from_files(nseq_list, nseq_list_sub, "/home/jg2447/slayman/motif_inference/result/EnrichmentGenomeShuffle/%s/enrichment/RankSPP_100.txt" % data_type)
    full_df = pd.DataFrame([data[0], data[1]]).transpose()
    full_df["sample"] = data_type.replace("human", "")
    full_df["rank"] = "SPP Shuffled"
    sub_df = pd.DataFrame([data[2], data[3], data[4]]).transpose()
    sub_df["sample"] = data_type.replace("human", "")
    sub_df["rank"] = "SPP Shuffled"
    full = pd.concat([full, full_df])
    sub = pd.concat([sub, sub_df])

    return((full, sub))

In [ ]:
# fix variables
nseq_list_sub = [50]+list(range(250, 3250, 250))
nseq_list = list(range(50, 3010, 10))
alpha_list = np.round(np.append(np.arange(0.1,1.0,0.1), np.arange(1.0,11.0,1.0)), decimals=2)

In [ ]:
full1, sub1 = get_data_sample("fly")
full2, sub2 = get_data_sample("worm")
full3, sub3 = get_data_sample("humanGM12878")
full4, sub4 = get_data_sample("humanK562")

full = pd.concat([full1, full2, full3, full4])
sub = pd.concat([sub1, sub2, sub3, sub4])

full.columns = ["top", "enrichment", "sample", "rank"]
sub.columns = ["top", "enrichment", "err", "sample", "rank"]

In [ ]:
# jit for plot
sub.loc[sub["rank"] == "C-score", "top"] += 10
sub.loc[sub["rank"] == "C-score Shuffled", "top"] += 10

In [ ]:
def get_data_sample_weight(data_type):
    data_hits = []
    enrichment = []
    for rank in rank_list:
        df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/%s/enrichment/%s_100.txt" % (data_type, rank), sep="\t", index_col=0)
        enrichment.append(df["250"].mean())

        data_hits.append(df["250"].values)
        if rank == "RankSPP":
            spp_hits = df["250"].values
        elif rank == "RankColdness":
            cold_hits = df["250"].values


    e_improve = (np.array(enrichment) - enrichment[0])/enrichment[0]
    e_improve_data = pd.DataFrame(e_improve).transpose()

    spp_p_list = []
    cold_p_list = []
    for i in range(len(data_hits)):
        spp_p_list.append(utils.stat_sig_test(data_hits[i], spp_hits))
        cold_p_list.append(utils.stat_sig_test(data_hits[i], cold_hits))

    data_p_list = []
    for i in range(len(spp_p_list)):
        if spp_p_list[i] < 0.05 and cold_p_list[i] < 0.05:
            data_p_list.append("*+")
        elif spp_p_list[i] < 0.05 and cold_p_list[i] >= 0.05:
            data_p_list.append("*")
        elif spp_p_list[i] >= 0.05 and cold_p_list[i] < 0.05:
            data_p_list.append("+")
        elif spp_p_list[i] >= 0.05 and cold_p_list[i] >= 0.05:
            data_p_list.append("")

    return((e_improve_data.values[0], np.array(data_p_list)))

In [ ]:
def get_data_sample_length(data_type):
    enrichment = []
    nbp_list = [10, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200]
    
    for nn in nbp_list:
        df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/%s/enrichment/RankSPP_%d.txt" % (data_type, nn), sep="\t", index_col=0)
        enrichment.append(df["250"].mean())
    e_len_improve = (np.array(enrichment) - enrichment[5])/enrichment[5]
    return(pd.DataFrame(e_len_improve))

In [ ]:
alpha_list_para = np.round(np.append(np.arange(0.1,1.0,0.1), np.arange(1.0,11.0,1.0)), decimals=2)
rank_list_para = ["RankLinear_%.1f" % i for i in alpha_list_para]

rank_list = ["RankSPP"]
rank_list.extend(rank_list_para)
rank_list.extend(["RankColdness"])

data1, p1 = get_data_sample_weight("fly")
data2, p2 = get_data_sample_weight("worm")
data3, p3 = get_data_sample_weight("humanGM12878")
data4, p4 = get_data_sample_weight("humanK562")

datalen1 = get_data_sample_length("fly")
datalen2 = get_data_sample_length("worm")
datalen3 = get_data_sample_length("humanGM12878")
datalen4 = get_data_sample_length("humanK562")

### plot

In [ ]:
figsize = (6.2, 6.4)
panel_number_fs = 8
x_tick_label_fs = 6
y_tick_label_fs = 6
x_label_fs = 7
y_label_fs = 7
title_fs = 7
legend_fs = 5

color = [
    (0.0,0.75,1.0,0.6),
    (0.8,0.2 ,0.4,0.4),
    (0.0,0.75,1.0,0.6),
    (0.8,0.2 ,0.4,0.4)]
color = ["#c2d2ed", "#ed6a6d", "#c2d2ed", "#ed6a6d"]

cmap = "RdBu"

In [ ]:
# plot a single line with error bar
def enrichment_line(ax, full, sub, color, sample):
    ax.plot("top", "enrichment", "-", data=full.query("sample == '%s' and rank == 'C-score'" % sample), color=color[0], lw=0.75)
    ax.plot("top", "enrichment", "-", data=full.query("sample == '%s' and rank == 'SPP'" % sample), color=color[1], lw=0.75)
    ax.plot("top", "enrichment", "--", data=full.query("sample == '%s' and rank == 'C-score Shuffled'" % sample), color=color[2], lw=0.75)
    ax.plot("top", "enrichment", "--", data=full.query("sample == '%s' and rank == 'SPP Shuffled'" % sample), color=color[3], lw=0.75)
    
    ax.xaxis.set_ticks([0, 1000, 2000, 3000])
    ax.tick_params(labelsize=x_tick_label_fs, pad=1, length=2.5, width=0.5)
    
    ax.spines['bottom'].set_linewidth(0.5)
    ax.spines['left'].set_linewidth(0.5)
        
    for i,r in enumerate(["C-score", "SPP", "C-score Shuffled", "SPP Shuffled"]):
        
        #### ribbon
#         y1 = sub.query("sample == '%s' and rank == '%s'" % (sample,r))["enrichment"].values + sub.query("sample == '%s' and rank == '%s'" % (sample,r))["err"].values
#         y2 = sub.query("sample == '%s' and rank == '%s'" % (sample,r))["enrichment"].values - sub.query("sample == '%s' and rank == '%s'" % (sample,r))["err"].values
#         ax.fill_between(
#             x = sub.query("sample == '%s' and rank == '%s'" % (sample,r))["top"],
#             y1 = y1,
#             y2 = y2,
#             color=color[i])
        
        #### error bar
        (_, caps, _) = ax.errorbar(
            x = sub.query("sample == '%s' and rank == '%s'" % (sample,r))["top"], 
            y = sub.query("sample == '%s' and rank == '%s'" % (sample,r))["enrichment"], 
            yerr = sub.query("sample == '%s' and rank == '%s'" % (sample,r))["err"], 
            fmt="o", ms=0.75, elinewidth=0.75, capsize=1, color=color[i])
        for cap in caps:
            cap.set_markeredgewidth(0)

In [ ]:
def plt_heatmap_weight(data, data_p, axs, ax_cbar, ylabeltext):
    # SPP
    ax1 = sns.heatmap(
            pd.DataFrame([data]).iloc[:, 0:1],
            ax=axs[0],
            vmax=abs(data).max(),
            vmin=-abs(data).max(),
            center=0,
            cmap=cmap,
            cbar=False,
            annot=pd.DataFrame([data_p]).iloc[:, 0:1],
            annot_kws={"fontsize":6},
            fmt="s",
            linewidths=.05)
    # BC-score
    ax2 = sns.heatmap(
            pd.DataFrame([data]).iloc[:, 1:-1],
            ax=axs[1],
            vmax=abs(data).max(),
            vmin=-abs(data).max(),
            center=0,
            cmap=cmap,
            cbar_ax=ax_cbar,
            cbar_kws={"orientation": "horizontal"},
            annot=pd.DataFrame([data_p]).iloc[:, 1:-1],
            annot_kws={"fontsize":6},
            fmt="s",
            linewidths=.05)
    # C-score
    ax3 = sns.heatmap(
            pd.DataFrame([data]).iloc[:, -1:],
            ax=axs[2],
            vmax=abs(data).max(),
            vmin=-abs(data).max(),
            center=0,
            cmap=cmap,
            cbar=False,
            annot=pd.DataFrame([data_p]).iloc[:, -1:],
            annot_kws={"fontsize":6},
            fmt="s",
            linewidths=.05)
    
    # add X label to only SPP square
    ax1.get_yaxis().set_ticklabels([ylabeltext], rotation=0, fontsize=y_tick_label_fs)
    ax1.get_yaxis().set_tick_params(size=0)
    ax2.get_yaxis().set_ticklabels([])
    ax2.get_yaxis().set_tick_params(size=0)
    ax3.get_yaxis().set_ticklabels([])
    ax3.get_yaxis().set_tick_params(size=0)
    
    # no X label
    ax1.xaxis.set_ticklabels([])
    ax1.xaxis.set_tick_params(size=0)
    ax2.xaxis.set_ticklabels([])
    ax2.xaxis.set_tick_params(size=0)
    ax3.xaxis.set_ticklabels([])
    ax3.xaxis.set_tick_params(size=0)
    
    # no color bar ticks
    ax_cbar.xaxis.set_ticklabels([])
    ax_cbar.xaxis.set_tick_params(size=0)
    
    # color bar label
    ax_cbar.annotate("%.2f" % (-abs(data).max()), xy=(-17,0.4), fontsize=legend_fs, xycoords="axes pixels")
    ax_cbar.annotate("%.2f" % (abs(data).max()), xy=(112,0.4), fontsize=legend_fs, xycoords="axes pixels")

    return((ax1, ax2, ax3))

In [ ]:
def plt_heatmap_length(ax, data, labeltext):
    ax = sns.heatmap(data,
                     ax=ax,
                     vmax=vmax,
                     vmin=-vmax,
                     center=0,
                     cmap=cmap,
                     cbar=False,
                     annot=data,
                     annot_kws={"fontsize":6},
                     fmt=".2f")
    
    ax.xaxis.set_ticklabels([])
    ax.xaxis.set_tick_params(size=0)
    
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    ax.yaxis.set_ticklabels([labeltext], fontsize=y_tick_label_fs, rotation=0)
    ax.yaxis.set_tick_params(size=0)

In [ ]:
# plot
sns.set_context("paper")

fig, axs = plt.subplots(
    nrows=2, ncols=1, 
    sharex=False, sharey=False,
    gridspec_kw={'hspace': 0.2, "height_ratios":[0.7, 2.5]},
    figsize=figsize, dpi=300)

axs[0].axis('off')
axs[1].axis('off')

########
gs = mpl.gridspec.GridSpecFromSubplotSpec(1, 4, subplot_spec=axs[0], hspace=0, wspace=0.35)

#### A
ax = fig.add_subplot(gs[0,0])
enrichment_line(ax, full, sub, color, sample="fly")
ax.yaxis.set_ticks([0.07, 0.14, 0.21, 0.28])
ax.set_ylim([0.07,0.28])
ax.set_ylabel("Motif enrichment", fontsize=y_label_fs)
ax.set_title("Fly", fontsize=title_fs)
ax.text(-0.3, 1.07, "A", transform=ax.transAxes, size=panel_number_fs, weight='bold')

#### B
ax = fig.add_subplot(gs[0,1])
enrichment_line(ax, full, sub, color, sample="worm")
ax.yaxis.set_ticks([0,0.08,0.16,0.24])
ax.set_ylim([0,0.24])
ax.set_title("Worm", fontsize=title_fs)
ax.text(-0.2, 1.07, "B", transform=ax.transAxes, size=panel_number_fs, weight='bold')
#### legend
ax_cbar = ax
ax_cbar.legend(
    handles=[
        (ax.lines[0], ax.lines[4]),
        (ax.lines[1], ax.lines[7]),
        (ax.lines[2], ax.lines[10]),
        (ax.lines[3], ax.lines[13])],
    labels=["C-score","SPP","C-score Ctrl","SPP Ctrl"],
    loc="upper left", fontsize=legend_fs, ncol=4, handlelength=2, bbox_to_anchor=(-0.2,1.42))

#### C
ax = fig.add_subplot(gs[0,2])
enrichment_line(ax, full, sub, color, sample="GM12878")
ax.yaxis.set_ticks([0.1, 0.25, 0.4, 0.55])
ax.set_ylim([0.075,0.575])
ax.set_xlabel("Binding sites ranking cutoff", fontsize=x_label_fs, x=-0.3)
ax.set_title("GM12878", fontsize=title_fs)
ax.text(-0.3, 1.07, "C", transform=ax.transAxes, size=panel_number_fs, weight='bold')

#### D
ax = fig.add_subplot(gs[0,3])
enrichment_line(ax, full, sub, color, sample="K562")
ax.yaxis.set_ticks([0.1, 0.25, 0.4, 0.55])
ax.set_ylim([0.075,0.575])
ax.set_title("K562", fontsize=title_fs)
ax.text(-0.2, 1.07, "D", transform=ax.transAxes, size=panel_number_fs, weight='bold')

sns.despine(trim=True)

################
gs = mpl.gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=axs[1], hspace=0.2, height_ratios=[1.5,1])
ax1 = fig.add_subplot(gs[0,0])
ax2 = fig.add_subplot(gs[1,0])
ax1.axis('off')
ax2.axis('off')
axs = [ax1, ax2]

################
gs = mpl.gridspec.GridSpecFromSubplotSpec(9, 5, subplot_spec=axs[0], hspace=0.4, wspace=0.05, height_ratios=[0.3,1,0.3,1,0.3,1,0.3,1,1.1], width_ratios=[1,6,7,6,1])

##
ax1 = fig.add_subplot(gs[1,0])
ax2 = fig.add_subplot(gs[1,1:4])
ax3 = fig.add_subplot(gs[1,4])
ax_cbar = fig.add_subplot(gs[0,2])
ax1, ax2, ax3 = plt_heatmap_weight(data1, p1, (ax1, ax2, ax3), ax_cbar, "Fly")
ax1.text(-1, 1.1, "E", transform=ax1.transAxes, size=panel_number_fs, weight='bold')

## 
ax1 = fig.add_subplot(gs[3,0])
ax2 = fig.add_subplot(gs[3,1:4])
ax3 = fig.add_subplot(gs[3,4])
ax_cbar = fig.add_subplot(gs[2,2])
ax1, ax2, ax3 = plt_heatmap_weight(data2, p2, (ax1, ax2, ax3), ax_cbar, "Worm")

##
ax1 = fig.add_subplot(gs[5,0])
ax2 = fig.add_subplot(gs[5,1:4])
ax3 = fig.add_subplot(gs[5,4])
ax_cbar = fig.add_subplot(gs[4,2])
ax1, ax2, ax3 = plt_heatmap_weight(data3, p3, (ax1, ax2, ax3), ax_cbar, "GM12878")

##
ax1 = fig.add_subplot(gs[7,0])
ax2 = fig.add_subplot(gs[7,1:4])
ax3 = fig.add_subplot(gs[7,4])
ax_cbar = fig.add_subplot(gs[6,2])
ax1, ax2, ax3 = plt_heatmap_weight(data4, p4, (ax1, ax2, ax3), ax_cbar, "K562")

# X labels
ax1.get_xaxis().set_ticklabels(["SPP"],
                               fontsize=x_tick_label_fs, rotation=90)
ax2.get_xaxis().set_ticklabels(["0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "1.0", "2.0", "3.0", "4.0", "5.0", "6.0", "7.0", "8.0", "9.0", "10.0"],
                               fontsize=x_tick_label_fs, rotation=90)
ax3.get_xaxis().set_ticklabels(["C-score"],
                               fontsize=x_tick_label_fs, rotation=90)
for axx in [ax1, ax2, ax3]:
    axx.get_xaxis().set_tick_params(size=2.5, pad=1, color="#757575")
ax2.set_xlabel("Binding site ranking method", fontsize=x_label_fs, labelpad=8.5)

ax_annot = fig.add_subplot(gs[8,1:4])
ax_annot.plot([1,21], [1,1], ",", color="white")
ax_annot.set_ylim([0,2])
ax_annot.set_xlim([0,21])    
ax_annot.annotate("", 
                  (0.5, 0.4),
                  (6.5, 0.4),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", 
                  (14.5, 0.4),
                  (20.5, 0.4),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", 
                  (0.5, 0),
                  (0.5, 0.8),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", 
                  (20.5, 0),
                  (20.5, 0.8),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("BC-score with different weight", 
                  (7.25, 0.25),
                  (7.25, 0.25),
                  rotation=0,
                  size=6)
ax_annot.xaxis.set_ticklabels([])
ax_annot.xaxis.set_tick_params(size=0)
ax_annot.axis('off')


################
gs = mpl.gridspec.GridSpecFromSubplotSpec(nrows=6, ncols=5, subplot_spec=axs[1], wspace=0.1, hspace=0.1, width_ratios=[0.5,1,1,1,0.5], height_ratios=[0.3, 0.2, 1, 1, 1, 1])

ax_cbar = fig.add_subplot(gs[0, 2])
vmax = max([datalen1.abs().max().values[0], datalen2.abs().max().values[0], datalen3.abs().max().values[0], datalen4.abs().max().values[0]])

##
ax = fig.add_subplot(gs[2, 1:4])
plt_heatmap_length(ax, datalen1.transpose(), "Fly")
ax.text(-0.15, 1.15, "F", transform=ax.transAxes, size=panel_number_fs, weight='bold')

##
ax = fig.add_subplot(gs[3, 1:4])
plt_heatmap_length(ax, datalen2.transpose(), "Worm")

##
ax = fig.add_subplot(gs[4, 1:4])
plt_heatmap_length(ax, datalen3.transpose(), "GM12878")

##
ax = fig.add_subplot(gs[5, 1:4])
ax = sns.heatmap(datalen4.transpose(),
                 ax=ax,
                 vmax=vmax,
                 vmin=-vmax,
                 center=0,
                 cmap=cmap,
                 cbar_ax=ax_cbar,
                 cbar_kws={"orientation":"horizontal"},
                 annot=datalen4.transpose(),
                 annot_kws={"fontsize":6},
                 fmt=".2f")
ax.yaxis.set_ticklabels(["K562"], fontsize=y_tick_label_fs, rotation=0)
ax.yaxis.set_tick_params(size=0)

ax.xaxis.set_ticklabels([20,40,80,120,160,200,240,280,320,360,400], rotation=0, fontsize=x_tick_label_fs)
ax.xaxis.set_tick_params(size=2.5, color="#757575", pad=1)
ax.set_xlabel("Binding site length", fontsize=8)

ax_cbar.xaxis.set_ticklabels([])
ax_cbar.xaxis.set_tick_params(size=0)
ax_cbar.annotate("%.2f" % (-vmax), xy=(-17,0.4), fontsize=legend_fs, xycoords="axes pixels")
ax_cbar.annotate("%.2f" % (vmax), xy=(81,0.4), fontsize=legend_fs, xycoords="axes pixels")

# plt.show()
plt.savefig("./fig2.pdf", dpi='figure', bbox_inches="tight")